# Doom in the weights — 192x120 on a Colab GPU

Runs the linuxformer LX circuit over a Doom trace and writes the frames the circuit verifies.

**Before running:** set `HF_TOKEN` (write access) and `HF_DATASET` in the Colab secrets panel
(key icon on the left), not in a cell. The dataset must already hold the artifacts listed in
`runs/README.md`.

References: `runs/README.md` for the budget and throughput numbers, `../STATE.md` for the full
decision trail, `../../CREDITS.md` for licenses.

In [ ]:
#@title 1. GPU check
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, "-c", "import torch;print('torch', torch.__version__, "
                      "'cuda', torch.version.cuda)"], capture_output=True, text=True).stdout)

## 2. Pin the stack

Do not skip this. The Colab image ships torch 2.14.1, and installing 2.11.0 over it corrupts the
package (`_EvalFrameOverride`). `colab_setup.py` removes torch completely, reinstalls, drops the
`torchao` metadata that `packaging` cannot parse, and adds `torchcodec`'s metadata because
`transformers 5.22` looks it up unconditionally at import time.

In [ ]:
#@title 2. Pin torch 2.11.0 / transformers 5.22 / fla 0.5.2 (~6 min)
import subprocess, sys
r = subprocess.run([sys.executable, "colab_setup.py"], capture_output=True, text=True)
print(r.stdout[-4000:])
print(r.stderr[-2000:])
assert "SETUP OK" in r.stdout, "setup failed -- read the output above before spending GPU time"

## 3. Fetch the artifacts

The circuit (`packed-192x120.pt`, ~4.1 GiB) is per-build: different `.text` bytes are a
different weight set. The driver checks the ELF's md5 against a known-good value and refuses to
run if it does not match — that check exists because applying one build's symbol offsets to
another build's program corrupts the guest and looks exactly like a broken circuit.

In [ ]:
#@title 3. Download and verify artifacts
import hashlib, os, pathlib
from google.colab import userdata
from huggingface_hub import hf_hub_download

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_DATASET"] = userdata.get("HF_DATASET")  # e.g. "<owner>/<dataset>"

ARTIFACTS = (("packed-192x120.pt", "packed.pt"),
             ("doom-lf-192x120.elf", "doom-lf.elf"),
             ("doom1.wad", "doom1.wad"),
             ("lxsrc.zip", "lxsrc.zip"),
             ("sched.txt", "sched.txt"))

DATA = pathlib.Path("/content/data-192")
DATA.mkdir(parents=True, exist_ok=True)
for src, dst in ARTIFACTS:
    got = hf_hub_download(os.environ["HF_DATASET"], src, repo_type="dataset",
                          local_dir=str(DATA), token=os.environ["HF_TOKEN"])
    if pathlib.Path(got).resolve() != (DATA / dst).resolve():
        (DATA / dst).write_bytes(pathlib.Path(got).read_bytes())
    p = DATA / dst
    h = hashlib.md5(p.read_bytes()).hexdigest()
    print(f"{src:24s} -> {dst:14s} {p.stat().st_size/2**20:7.1f} MiB  md5 {h}")

elf_md5 = hashlib.md5((DATA / "doom-lf.elf").read_bytes()).hexdigest()
assert elf_md5 == "ff4ad5641ef0e7a000ed5536a6cddd93", f"wrong ELF: {elf_md5}"
print("ELF matches the 192x120 build")

## 4. Measure before committing the budget

`--bench` runs the warm-up chunk (which includes `torch.compile`) and stops without writing
anything. The number it prints is *not* steady state — compilation dominates — but the run's own
progress lines are. Read the first 1M-token line and size `NTOK` from that.

Budget: 192x120 needs **~106M tokens** to reach frame 121 (boot 7.6M + 116 frames x 694k).
A 95M budget stops at frame 119 and needs a resume.

In [ ]:
#@title 4. Run the verifier (3h cap, resumes from results/session-192.pt if present)
import os, subprocess, sys
env = dict(os.environ, SET="192", RESUME=os.environ.get("RESUME", "0"),
           HOURS="3.0", NTOK="95000000")
log = open("/content/run.log", "w")
p = subprocess.Popen([sys.executable, "colab_run.py"], stdout=log,
                     stderr=subprocess.STDOUT, env=env)
open("/content/job.pid", "w").write(str(p.pid))
print("launched pid", p.pid, "-- poll /content/run.log; the CLI drops a silent kernel")

In [ ]:
#@title 5. Poll (run this cell repeatedly; it returns immediately)
import subprocess
print(subprocess.run(["bash", "-lc",
  "(kill -0 $(cat /content/job.pid) 2>/dev/null && echo running) || echo finished; "
  "echo frames=$(ls /content/frames 2>/dev/null | wc -l); tail -6 /content/run.log"],
  capture_output=True, text=True).stdout)

## 6. Check the frames against the reference machine

This is the only claim that matters: each frame's PPM payload must equal `fastemu-lf`'s. A frame
is written only when every token it is made of has been verified, so equality here means the
circuit predicted the whole trace that produced those pixels.

`fastemu-lf` itself is CPU-only and takes about half a second for 130 frames, so it can run in
this notebook, or locally before the GPU job.

In [ ]:
#@title 6. Rebuild one reference frame with fastemu-lf and compare
import glob, os, subprocess

# fastemu-lf builds from the port submodule: cc -O3 -o fastemu-lf port/native/fastemu-lf.c -lm
REF = "/content/ref"
os.makedirs(REF, exist_ok=True)
subprocess.run(["/content/fastemu-lf", "/content/data-192/doom-lf.elf",
                "--wad", "/content/data-192/doom1.wad",
                "--fbp-sym", "DG_ScreenBuffer", "--fc-sym", "qwen_frame_count",
                "--keys-sym", "qwen_key_state", "--demo-sym", "qwen_demo_enabled",
                "--frames", "122", "--script", "/content/data-192/sched.txt",
                "--dump-every", "1", "--dump-from", "6", "--dump-to", "121",
                "--dump-dir", REF, "--guest-out", "none"], check=True)

def payload(p):
    d = open(p, "rb").read()
    return d[d.index(b"255\n") + 4:]

W, H = 192, 120
n = 192 * 120 * 3
ok = bad = 0
for f in sorted(glob.glob("/content/frames/frame-*.ppm")):
    ref = os.path.join(REF, os.path.basename(f))
    if not os.path.exists(ref):
        continue
    if payload(ref)[:n] == payload(f):
        ok += 1
    else:
        bad += 1
        print("DIFFERS:", os.path.basename(f))
print(f"byte-exact vs fastemu-lf: {ok} ok, {bad} differing")

## 7. Release the GPU

The runtime bills while it exists. Stop it from the CLI (`colab stop -s <session>`) or from
*Runtime → Disconnect and delete runtime*. The checkpoint and frames are already on the dataset,
so nothing is lost.